# 6. tēma — dzīvokļu sludinājumu datu ieguve no tīmekļa

[![Atvērt Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ValRCS/RTU_BDAA_Course_2026/blob/main/notebooks/lecture_06_web_scraping/06_02_apartment_scraping.ipynb)

Šajā darba burtnīcā izmantosim **SS.com dzīvokļu sludinājumus** kā reālāku web scraping piemēru.

Mērķis nav izveidot industriālu SS.com datu vācēju. Mērķis ir saprast tipisku biznesa datu ieguves procesu:

**vietne → HTTP → HTML tabula → DataFrame → neliela tīrīšana → kopsavilkums → CSV**

Pēc darba burtnīcas jūs pratīsiet:

- lejupielādēt reālas rezultātu lapas HTML;
- pārbaudīt, kādas tabulas lapā ir atrodamas;
- izmantot `pandas.read_html()` strukturētu HTML tabulu nolasīšanai;
- ar BeautifulSoup atrast sludinājumu saites;
- veikt nelielu cenu un platību datu normalizāciju;
- saglabāt iegūtos datus turpmākai analīzei.

> Vietņu HTML struktūra laika gaitā mainās. Ja kāds selektors vairs nedarbojas, tas ir normāls web scraping dzīves cikla piemērs, nevis Python kļūda.


## Darbināšana VS Code un Google Colab

Darba burtnīca ir veidota tā, lai tā darbotos abās vidēs.

### VS Code
- Python;
- VS Code **Python** paplašinājums;
- VS Code **Jupyter** paplašinājums;
- izvēlēts Python kernel.

### Google Colab
Atveriet notebook ar **Open in Colab** pogu augstāk.

Sagatavošanas šūna automātiski instalē tikai tās bibliotēkas, kuru konkrētajā vidē trūkst.


In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "requests": "requests",
    "bs4": "beautifulsoup4",
    "pandas": "pandas",
    "lxml": "lxml",
}

missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Instalējam trūkstošās pakotnes:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Visas nepieciešamās pakotnes jau ir instalētas.")

print("Python:", sys.version.split()[0])
print("Vide:", "Google Colab" if "google.colab" in sys.modules else "lokāls Jupyter/VS Code")


In [ ]:
from io import StringIO
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; RTU-BDAA-teaching-example/1.0)"
}

# Varat nomainīt rajonu vai pārdošanu/īri.
URL = "https://www.ss.com/en/real-estate/flats/riga/centre/sell/"

print(URL)


## 1. Iegūstam HTML ar `requests`

Salīdzinot ar `pd.read_html(URL)`, ir lietderīgi HTML vispirms paņemt pašiem:

- varam norādīt `User-Agent`;
- varam pārbaudīt statusa kodu;
- varam izmantot **to pašu HTML** gan BeautifulSoup, gan Pandas;
- atkļūdošana kļūst saprotamāka.


In [ ]:
SAMPLE_HTML = """
<!doctype html>
<html>
<head><title>SS.com apartment sample for classroom fallback</title></head>
<body>
<table>
<thead>
<tr>
<th>Advertisements date</th><th>Street</th><th>R.</th><th>m²</th><th>Floor</th>
<th>Series</th><th>Price, m2</th><th>Price</th>
</tr>
</thead>
<tbody>
<tr><td>Sample listing A</td><td>Kristapa 2</td><td>3</td><td>81</td><td>1/2</td><td>Recon.</td><td>2,963 €</td><td>240,000 €</td></tr>
<tr><td>Sample listing B</td><td>Baldones 28</td><td>2</td><td>40</td><td>4/5</td><td>Lit pr.</td><td>1,722 €</td><td>68,888 €</td></tr>
<tr><td>Sample listing C</td><td>Ranka d. 9</td><td>2</td><td>54</td><td>2/6</td><td>Recon.</td><td>2,315 €</td><td>125,000 €</td></tr>
<tr><td>Sample listing D</td><td>Darba 11a</td><td>1</td><td>29</td><td>1/2</td><td>Stalin project</td><td>1,103 €</td><td>32,000 €</td></tr>
<tr><td>Sample listing E</td><td>M. Nometnu 11</td><td>1</td><td>40</td><td>1/2</td><td>Pre-war house</td><td>925 €</td><td>37,000 €</td></tr>
</tbody>
</table>
</body>
</html>
"""

try:
    response = requests.get(URL, headers=HEADERS, timeout=20)
    response.raise_for_status()
    html = response.text
    source_mode = "live SS.com"

    print("Status:", response.status_code)
    print("Content-Type:", response.headers.get("content-type"))
    print("HTML garums:", len(html))
except requests.RequestException as exc:
    # The notebook remains runnable in both Colab and local Jupyter even if
    # the live site temporarily blocks cloud IPs or is unavailable.
    print("Dzīvo SS.com lapu neizdevās saņemt:", exc)
    print("Turpinām ar nelielu mācību HTML paraugu.")
    html = SAMPLE_HTML
    source_mode = "embedded classroom sample"

print("Datu avots:", source_mode)
print("HTML sākums:")
print(html[:250])


### Ja vietne bloķē pieprasījumu

Publiska vietne var mainīt aizsardzības vai piekļuves noteikumus. Ja saņemat `403`, `429` vai citu kļūdu:

1. nepalaidiet pieprasījumu ciklā atkārtoti;
2. pārbaudiet URL pārlūkā;
3. apskatiet vietnes noteikumus un `robots.txt`;
4. mācību stundā izmantojiet pasniedzēja saglabātu HTML/CSV paraugu, ja dzīvais avots konkrētajā brīdī nav pieejams.

**429 Too Many Requests** īpaši nozīmē, ka jāsamazina pieprasījumu biežums.


## 2. Apskatām HTML ar BeautifulSoup

Vispirms noskaidrosim, cik `<table>` elementu lapa satur.

SS.com vēsturiski ir izmantojis HTML tabulas arī lapas izkārtojumam, tāpēc lapā var būt vairākas tabulas.


In [ ]:
soup = BeautifulSoup(html, "html.parser")

print("Lapas title:", soup.title.get_text(" ", strip=True) if soup.title else "(nav title)")
print("HTML tabulu skaits:", len(soup.find_all("table")))
print("Saišu skaits:", len(soup.find_all("a")))


## 3. Strukturētas HTML tabulas ar `pandas.read_html()`

Ja tīmekļa lapā dati jau ir `<table>` struktūrā, nav nepieciešams manuāli lasīt katru `<td>` elementu.

`pd.read_html()`:

- atrod HTML tabulas;
- pārvērš tās DataFrame objektos;
- atgriež **DataFrame sarakstu**.

Svarīgi: tabulas numurs nav stabils API. Tāpēc nepaļaujamies tikai uz `tables[4]`; mēģināsim atrast sludinājumu tabulu pēc kolonnu nosaukumiem.


In [ ]:
try:
    tables = pd.read_html(StringIO(html))
except ValueError:
    tables = []

print("Atrastas tabulas:", len(tables))

for i, table in enumerate(tables):
    columns = [str(c) for c in table.columns]
    print(f"{i}: shape={table.shape}, columns={columns[:12]}")


## 4. Atrodam sludinājumu tabulu

SS.com angļu versijas dzīvokļu tabulā parasti parādās kolonnas, piemēram:

- `Street`
- `R.`
- `m²`
- `Floor`
- `Series`
- `Price, m2`
- `Price`

Izvēlēsimies pirmo tabulu, kurā atrodam gan `Street`, gan kādu `Price` kolonnu.

Šī pieeja ir izturīgāka par konkrēta tabulas indeksa ierakstīšanu kodā.


In [ ]:
def flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()

    if isinstance(result.columns, pd.MultiIndex):
        result.columns = [
            " ".join(str(part) for part in col if str(part) != "nan").strip()
            for col in result.columns
        ]
    else:
        result.columns = [str(col).strip() for col in result.columns]

    return result


def find_listing_table(table_list):
    for table in table_list:
        candidate = flatten_columns(table)
        names = [str(c).lower() for c in candidate.columns]

        has_street = any("street" in c for c in names)
        has_price = any("price" in c for c in names)

        if has_street and has_price:
            return candidate
    return None


listing_df = find_listing_table(tables)

if listing_df is None:
    print(
        "Dzīvajā HTML sludinājumu tabulu neatradām. "
        "Iespējams, vietnes struktūra ir mainījusies vai saņemta aizsardzības lapa."
    )
    print("Turpinām ar iebūvēto mācību paraugu, lai pārējās šūnas būtu izpildāmas.")

    html = SAMPLE_HTML
    source_mode = "embedded classroom sample"
    soup = BeautifulSoup(html, "html.parser")
    tables = pd.read_html(StringIO(html))
    listing_df = find_listing_table(tables)

print("Datu avots:", source_mode)
print("Izvēlētās tabulas forma:", listing_df.shape)
display(listing_df.head())


## 5. Ātra datu pārbaude

Pirms tīrīšanas vienmēr paskatāmies:

- tabulas izmēru;
- kolonnu nosaukumus;
- pirmās rindas;
- datu tipus;
- trūkstošās vērtības.

Šeit sākas pāreja no **datu ieguves** uz **datu apstrādi**.
Detalizētu tīrīšanu turpināsim nākamajā lekcijā.


In [ ]:
print("Shape:", listing_df.shape)
print()
print("Columns:")
print(listing_df.columns.tolist())
print()
listing_df.info()


## 6. Neliela tīrīšana: cena un platība

No tīmekļa iegūtas skaitliskas vērtības bieži ir teksta formā:

- `240,000 €`
- `2,963 €`
- `81`

Analīzei vēlamies `int` vai `float`.

Izveidosim palīgfunkciju, kas no teksta atstāj tikai ciparus un pēc tam izmanto `pd.to_numeric()`.


In [ ]:
def digits_to_number(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype("string")
        .str.replace(r"[^0-9]", "", regex=True)
        .replace("", pd.NA)
    )
    return pd.to_numeric(cleaned, errors="coerce")


apartments = listing_df.copy()

price_col = next(
    (c for c in apartments.columns if str(c).strip().lower() == "price"),
    None,
)
area_col = next(
    (c for c in apartments.columns if str(c).strip().lower() in {"m²", "m2"}),
    None,
)

if price_col:
    apartments["price_eur"] = digits_to_number(apartments[price_col])

if area_col:
    apartments["area_m2"] = pd.to_numeric(apartments[area_col], errors="coerce")

display(apartments.head())


### Piezīme par cenu formātiem

Šī vienkāršā funkcija ir piemērota **pārdošanas** sludinājumiem, kuros gala cena parasti ir vesels EUR skaitlis.

Īres sludinājumos var parādīties, piemēram, `430 €/mon.`.  
Tur skaitļa iegūšana vēl darbojas, bet papildus būtu jāsaglabā arī **periods** (`month`, `day` u.c.).

Tas ir labs piemērs, kāpēc reālos datos nepietiek tikai ar `replace("€", "")`.


## 7. Vienkāršs kopsavilkums

Vēl neveidosim pilnu datu analīzi — tā būs nākamās tēmas galvenā daļa.

Tomēr jau tagad varam pārbaudīt, vai scraping rezultāts ir jēgpilns.


In [ ]:
if "price_eur" in apartments.columns:
    display(apartments["price_eur"].describe())

if "area_m2" in apartments.columns:
    display(apartments["area_m2"].describe())


In [ ]:
if {"price_eur", "area_m2"}.issubset(apartments.columns):
    apartments["calculated_eur_per_m2"] = (
        apartments["price_eur"] / apartments["area_m2"]
    ).round(2)

    useful_columns = [
        c for c in ["Street", "R.", "area_m2", "price_eur", "calculated_eur_per_m2"]
        if c in apartments.columns
    ]
    display(
        apartments[useful_columns]
        .dropna(subset=["price_eur", "area_m2"])
        .sort_values("calculated_eur_per_m2")
        .head(10)
    )


## 8. Sludinājumu saites ar BeautifulSoup

`read_html()` ir lielisks tabulām, bet tas ne vienmēr saglabā `<a href="...">` saites.

Tāpēc praksē ir normāli kombinēt:

- **Pandas** tabulas struktūrai;
- **BeautifulSoup** HTML atribūtiem un saitēm.

SS.com sludinājumu saites parasti norāda uz URL, kas satur `/msg/`.


In [ ]:
ad_links = []

for a in soup.find_all("a", href=True):
    href = a.get("href")
    if "/msg/" in href:
        full_url = urljoin(URL, href)
        text = a.get_text(" ", strip=True)

        ad_links.append({
            "link_text": text,
            "url": full_url,
        })

links_df = pd.DataFrame(ad_links).drop_duplicates(subset=["url"])

print("Unikālas sludinājumu saites:", len(links_df))
display(links_df.head(10))


## 9. Kāpēc tabulas rindas un saites nav automātiski jāsavieno pēc indeksa?

Var rasties kārdinājums darīt:

```python
apartments["url"] = links_df["url"]
```

Tas ir droši tikai tad, ja esam pārbaudījuši, ka:

- abu tabulu rindu skaits sakrīt;
- HTML elementu secība tiešām atbilst DataFrame rindām;
- nav papildu reklāmas vai navigācijas saišu.

Profesionālā scraping risinājumā saiti labāk iegūt no **tās pašas sludinājuma rindas**, kuru parsējam, vai izmantot stabilu ID kā savienošanas atslēgu.

Šajā lekcijā svarīgākais ir saprast atšķirību starp **tekstu/tabulu datiem** un **HTML atribūtiem**.


## 10. Saglabājam rezultātu

Saglabājam gan tabulu, gan atrastās saites atsevišķos CSV failos.

Nākamajā lekcijā šo iegūto datu kopu var izmantot tīrīšanai un vizualizācijai.


In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

apartments_file = OUTPUT_DIR / "ss_apartments_raw_and_lightly_cleaned.csv"
links_file = OUTPUT_DIR / "ss_apartment_links.csv"

apartments.to_csv(apartments_file, index=False, encoding="utf-8-sig")
links_df.to_csv(links_file, index=False, encoding="utf-8-sig")

print("Saglabāts:")
print("-", apartments_file.resolve())
print("-", links_file.resolve())


## 11. Papildu uzdevums — cits rajons

Mainiet tikai URL, piemēram:

```python
https://www.ss.com/en/real-estate/flats/riga/agenskalns/sell/
https://www.ss.com/en/real-estate/flats/riga/purvciems/sell/
https://www.ss.com/en/real-estate/flats/riga/centre/hand_over/
```

Pārbaudiet:

1. vai tabulas struktūra saglabājas;
2. vai mainās kolonnu saturs;
3. kā atšķiras cenas;
4. kas notiek ar cenu parsēšanu īres sludinājumos.


## 12. Papildu uzdevums — vairākas rezultātu lapas

Ja vietnei ir vairākas lapas, scraping kļūst par atkārtojamu procesu:

1. atrod nākamās lapas saiti;
2. pieprasa nākamo lapu;
3. nolasa tabulu;
4. pievieno DataFrame sarakstam;
5. beigās izmanto `pd.concat()`.

Svarīgi:

- neveidojiet ātru bezgalīgu ciklu;
- ievietojiet pauzi starp pieprasījumiem;
- ierobežojiet lapu skaitu mācību eksperimentā;
- beidziet, ja nav nākamās lapas;
- apstrādājiet tīkla kļūdas.

Piemērs idejai:

```python
frames = []

for url in urls:
    # request
    # read_html
    # find listing table
    frames.append(df)
    time.sleep(1)

combined = pd.concat(frames, ignore_index=True)
```

Šo uzdevumu ir vērts pabeigt tikai pēc tam, kad viena lapa darbojas korekti.


## Noslēgums

Šajā lekcijā izmantojām divus savstarpēji papildinošus paņēmienus:

### BeautifulSoup
Labs, ja jāstrādā ar:

- HTML elementiem;
- atribūtiem;
- saitēm;
- nevienmērīgu lapas struktūru.

### `pandas.read_html()`
Ļoti ērts, ja dati jau atrodas HTML `<table>`.

Tipiska darba plūsma:

**requests → HTML → BeautifulSoup / read_html → DataFrame → neliela normalizācija → CSV**

Nākamajā tēmā galvenais jautājums vairs nebūs **“kā dabūt datus?”**, bet gan:

**“kā nekārtīgus iegūtos datus pārveidot uzticamai analīzei un vizualizācijai?”**
